# Agent-Sandbox SWE-bench Integration: Warmpool Strategies Demo

This notebook demonstrates how to configure and run the DeepSWE evaluation with different GKE Agent-Sandbox warmpool orchestration strategies.

For details on the architecture and design, see the [Design Document](file:///google/src/cloud/glottman/cliagent_sandbox_integration/google3/.plans/agent_sandbox_swe_bench_design.md).
For a quick CLI guide, see the [README](file:///google/src/cloud/glottman/cliagent_sandbox_integration/google3/.plans/README.md).

In [ ]:
import os
import subprocess
import sys

# Verify we are in the google3 workspace
cwd = os.getcwd()
print(f"Current working directory: {cwd}")
if "google3" not in cwd:
    print("WARNING: You should run this notebook from within the google3 workspace directory.")

## Common Configuration
Define the common settings for the evaluation, such as the model, split, and limits.
We use a small task limit (`TASKS_LIMIT=3`) for this demonstration to avoid long execution times and resource exhaustion.

In [ ]:
COMMON_ENV = {
    "MODEL_VERSION": "Qwen/Qwen3-32B",
    "DATASET_NAME": "R2E-Gym/SWE-Bench-Verified",
    "DATASET_SPLIT": "test",
    "TASKS_LIMIT": "3",          # Small limit for testing
    "MAX_CONCURRENT": "2",       # Concurrency limit
    "NODE_SELECTOR_KEY": "cloud.google.com/gke-nodepool",
    "NODE_SELECTOR_VAL": "deepswe-cpu-pool",
}

## Strategy 1: No Warmpools (Default)
In this mode, sandboxes are created on-demand for each task. There is no pre-warming.
*   **Use case:** Debugging, or when resource budget is very tight and latency is not a priority.
*   **Behavior:** Each task incurs a cold-start delay while the sandbox pod is provisioned.

In [ ]:
env = COMMON_ENV.copy()
env.update({
    "BACKEND": "kubernetes-sandbox",
    "WARMPOOL_STRATEGY": "none",
})

print("Running evaluation with NO warmpools...")
# Note: In a real run, this might fail if JAX/TPU is not configured.
# We run it with --help or check imports to verify the entry point if we want a quick check,
# but here we show the command to run the full eval.
cmd = ["python3", "third_party/py/tunix/oss/examples/deepswe/eval_deepswe.py"]
print(f"Command: {' '.join(cmd)} with env {env}")

# Uncomment to run (requires cluster access and configured environment)
# result = subprocess.run(cmd, env={**os.environ, **env}, capture_output=True, text=True)
# print(result.stdout)
# print(result.stderr, file=sys.stderr)

## Strategy 2: Naive Parallel Warmpools
Creates warmpools for all unique images in the dataset at the start of the job.
*   **Use case:** Small datasets, or when you have sufficient cluster capacity to pre-allocate warmpods for all images concurrently.
*   **Behavior:** Fast startup for all tasks, but high initial resource usage.

In [ ]:
env = COMMON_ENV.copy()
env.update({
    "BACKEND": "kubernetes-sandbox",
    "WARMPOOL_STRATEGY": "naive",
    "MAX_WARMPOOL_SIZE": "5",  # Pre-warm up to 5 pods per image
})

print("Running evaluation with Naive Parallel warmpools...")
cmd = ["python3", "third_party/py/tunix/oss/examples/deepswe/eval_deepswe.py"]
print(f"Command: {' '.join(cmd)} with env {env}")

# Uncomment to run
# result = subprocess.run(cmd, env={**os.environ, **env}, capture_output=True, text=True)
# print(result.stdout)

## Strategy 3: Sliding Window Warmpools (Recommended)
Sorts tasks by image and maintains a sliding window of active warmpools.
*   **Use case:** Large-scale evaluations with many different images and limited cluster capacity.
*   **Behavior:** Dynamically creates warmpools for upcoming images in the queue and deletes them when all tasks for that image are completed. Balance between startup speed and resource usage.

In [ ]:
env = COMMON_ENV.copy()
env.update({
    "BACKEND": "kubernetes-sandbox",
    "WARMPOOL_STRATEGY": "sliding",
    "WARMPOOL_WINDOW_SIZE": "2",  # Keep warmpools for 2 unique images active
    "MAX_WARMPOOL_SIZE": "5",     # Max 5 pre-warmed pods per image pool
})

print("Running evaluation with Sliding Window warmpools...")
cmd = ["python3", "third_party/py/tunix/oss/examples/deepswe/eval_deepswe.py"]
print(f"Command: {' '.join(cmd)} with env {env}")

# Uncomment to run
# result = subprocess.run(cmd, env={**os.environ, **env}, capture_output=True, text=True)
# print(result.stdout)